# Team 02 - Data Preprocessing Notebook

**Team:** Data Preprocessing  
**Worker name:** _fill in_  
**Date:** _fill in_  
**Dataset assigned:** _e.g. thaillm_  
**Shard range:** _e.g. shard_000 - shard_009_


## Step 1 - Environment Setup

In [ ]:
import sys
sys.path.insert(0, "../../code/preprocess")
sys.path.insert(0, "../../code/validate")

from pathlib import Path
import gzip, json

# Adjust these paths to your environment
INPUT_DIR  = Path("/mnt/ssd/data/thaillm")    # raw SSD data
OUTPUT_DIR = Path("/mnt/ssd/shards/thaillm")  # processed shards output
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ASSIGNED_SOURCE = "thaillm"  # change per your assignment
print("Setup OK")

## Step 2 - Run Schema Validation on Raw Shard

In [ ]:
import subprocess

SHARD = "shard_000.jsonl.gz"  # change per shard you are working on
raw_path = INPUT_DIR / SHARD

result = subprocess.run(
    [sys.executable, "../../code/validate/validator.py", str(raw_path)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("FAIL - fix errors above before continuing")
else:
    print("PASS - proceed to conversion")

## Step 3 - Format Conversion (convert_to_schema.py)

Spam filter runs automatically inside this step. Output: clean shard + `.spam.jsonl.gz` sidecar.

In [ ]:
output_path = OUTPUT_DIR / SHARD

result = subprocess.run(
    [
        sys.executable, "../../code/preprocess/convert_to_schema.py",
        "--source", ASSIGNED_SOURCE,
        "--input",  str(raw_path),
        "--output", str(output_path),
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## Step 4 - Check Spam Sidecar

In [ ]:
spam_path = output_path.with_suffix("").with_suffix(".spam.jsonl.gz")

if spam_path.exists():
    with gzip.open(spam_path, "rt") as f:
        spam_docs = [json.loads(l) for l in f]
    
    with gzip.open(output_path, "rt") as f:
        clean_count = sum(1 for _ in f)
    
    spam_rate = len(spam_docs) / max(clean_count + len(spam_docs), 1)
    print(f"Clean docs : {clean_count:,}")
    print(f"Spam docs  : {len(spam_docs):,}")
    print(f"Spam rate  : {spam_rate:.1%}")
    print()
    if spam_rate > 0.20:
        print("!! ESCALATE TO CALVIN - spam rate exceeds 20%")
    elif spam_rate > 0.05:
        print("Spot-check required (rate 5-20%) - review 10 docs below")
        for i, doc in enumerate(spam_docs[:10]):
            print(f"  [{i}] reasons={doc['_spam_reasons']} | text={doc['text'][:80]}")
    else:
        print("OK - log count in Google Sheet and continue")
else:
    print("No spam sidecar produced - all docs clean")

## Step 5 - Run Final Validator on Output Shard

In [ ]:
result = subprocess.run(
    [sys.executable, "../../code/validate/validator.py", str(output_path)],
    capture_output=True, text=True
)
print(result.stdout)
VALIDATOR_RESULT = "PASS" if result.returncode == 0 else "FAIL"

## Step 6 - Human Spot-Check (50 Random Docs)

In [ ]:
import random

with gzip.open(output_path, "rt", encoding="utf-8") as f:
    all_docs = [json.loads(l) for l in f]

sample = random.sample(all_docs, min(50, len(all_docs)))
defects = 0

for i, doc in enumerate(sample[:10]):  # print first 10 for review
    print(f"--- {i} ---")
    print(f"id     : {doc['id']}")
    print(f"source : {doc['source']} | domain: {doc['domain']}")
    print(f"flags  : {doc['quality_flags']}")
    print(f"text   : {doc['text'][:120]}")
    print()

# Update defect count manually after reviewing
defects = 0  # <- set this after your review
defect_rate = defects / len(sample)
print(f"Defect rate: {defect_rate:.1%} ({'PASS' if defect_rate < 0.02 else 'FAIL - escalate'})")

## Step 7 - Log Summary Row (paste into Google Sheet)

In [ ]:
import hashlib

sha256 = hashlib.sha256(open(output_path, "rb").read()).hexdigest()

log_row = {
    "worker":         "",  # your name
    "shard":          SHARD,
    "source":         ASSIGNED_SOURCE,
    "n_docs":         len(all_docs),
    "n_spam":         len(spam_docs) if spam_path.exists() else 0,
    "validator":      VALIDATOR_RESULT,
    "defect_rate":    f"{defect_rate:.1%}",
    "sha256":         sha256[:16] + "...",
    "output_path":    str(output_path),
}

print(json.dumps(log_row, indent=2, ensure_ascii=False))